# 02 - Centralized bridge: MLP vs XGBoost pada 9 fitur SFM (Paper 4, Fase 1)

**Tujuan:** JEMBATAN ke Paper 1. Pastikan **MLP** (backbone FL Paper 4) tidak kalah
jauh dari **XGBoost** (backbone Paper 1/2) pada 9 fitur SFM, dalam setting
**centralized** (data digabung — HANYA sebagai upper bound/sanity, bukan skenario
privasi). Kalau MLP centralized ~ XGBoost centralized, maka memakai MLP untuk FL
adalah pilihan yang sah (tak mengorbankan kualitas backbone).

**Output:** `fed_out/bridge_centralized.csv` (MCC/recall/precision/F1/bal-acc,
XGBoost vs MLP, dua arah evaluasi). Memuat partisi dari nb01 (`fed_out/*.npz`).

> Jalankan di SageMaker setelah nb01. JANGAN mengarang angka.

## 0. Setup

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn','xgboost'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import matthews_corrcoef, recall_score, precision_score, f1_score, balanced_accuracy_score
from xgboost import XGBClassifier
OUTDIR='fed_out'; SEED=42
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p, allow_pickle=True); return d['X'], d['y']
print('setup siap')

## 1. Muat partisi SFM (dari nb01) & gabung centralized

Centralized = gabungan train CIC + UNSW (HANYA utk upper bound). Evaluasi pada
test masing-masing + test gabungan (held-out).

In [ ]:
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
Xc_tr,yc_tr=load('cic_train.npz');  Xc_te,yc_te=load('cic_test.npz')
ok = all(v is not None for v in [Xu_tr,Xc_tr,Xu_te,Xc_te])
if ok:
    Xtr=np.vstack([Xc_tr,Xu_tr]); ytr=np.concatenate([yc_tr,yu_tr])
    print('centralized train', Xtr.shape, 'pos-rate', round(float(ytr.mean()),3))
else:
    print('(partisi belum ada - jalankan nb01 dulu di SageMaker)')

## 2. Latih XGBoost & MLP (centralized) + evaluasi

XGBoost: config Paper 1/2 (max_depth=8, lr=0.1, n_estimators=200, subsample 0.8).
MLP: 9->[64,32]->1, config draf Paper 4 (lihat dokumentasi Bagian B).

In [ ]:
def metrics(y,yp):
    return dict(MCC=round(matthews_corrcoef(y,yp),4), recall=round(recall_score(y,yp,zero_division=0),4),
                precision=round(precision_score(y,yp,zero_division=0),4), F1=round(f1_score(y,yp,zero_division=0),4),
                bal_acc=round(balanced_accuracy_score(y,yp),4))
res=[]
if ok:
    sc=StandardScaler().fit(Xtr)                 # centralized scaler (upper-bound saja)
    Ztr=sc.transform(Xtr)
    xgb=XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=SEED,tree_method='hist').fit(Xtr,ytr)
    # RUN-2 FIX: MLP under-trained di run-1 (ConvergenceWarning). Naikkan kapasitas+iterasi
    # + early_stopping agar konvergen -> perbandingan MLP vs XGBoost adil.
    mlp=MLPClassifier(hidden_layer_sizes=(128,64),activation='relu',alpha=1e-4,
        batch_size=256,learning_rate_init=1e-3,max_iter=500,early_stopping=True,
        n_iter_no_change=20,random_state=SEED).fit(Ztr,ytr)
    evalsets={'CIC-test':(Xc_te,yc_te),'UNSW-test':(Xu_te,yu_te),
              'GLOBAL-test':(np.vstack([Xc_te,Xu_te]),np.concatenate([yc_te,yu_te]))}
    for name,(Xe,ye) in evalsets.items():
        res.append({'eval':name,'model':'XGBoost', **metrics(ye, xgb.predict(Xe))})
        res.append({'eval':name,'model':'MLP',     **metrics(ye, mlp.predict(sc.transform(Xe)))})
    dfres=pd.DataFrame(res); dfres.to_csv(os.path.join(OUTDIR,'bridge_centralized.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    print('CEK: MLP GLOBAL-test MCC vs XGBoost GLOBAL-test MCC -> gap kecil = MLP layak utk FL.')
else:
    print('(lewati: partisi belum ada)')
print('=== SEL 2 (bridge) SELESAI ===')

## 3. Catatan untuk naskah

- Ini centralized **upper bound** + jembatan backbone; BUKAN hasil federated.
- Bila MLP MCC jauh di bawah XGBoost, naikkan kapasitas/epoch MLP secukupnya & catat;
  tujuannya MLP cukup kompetitif agar perbandingan FL adil.
- Scaler centralized di sini hanya untuk upper-bound; di FL (Fase 2) z-score **per-klien**.
- Hasil -> tabel D (baseline) di `documentation_federated.md`.